# Entraînement v2 — Legal-BERT, poids de classes, 512 tokens (Colab GPU)

**Objectif :** améliorer le modèle v1 (`bert-base-uncased`, 2 epochs, 256 tokens, entraîné par Charles dans `colab.ipynb`),
surtout le **F1 macro**, plombé par les classes rares (9 classes à F1 = 0).

| Levier | v1 | v2 |
|---|---|---|
| Modèle de départ | `bert-base-uncased` | `nlpaueb/legal-bert-base-uncased` (pré-entraîné sur des textes juridiques) |
| Longueur max | 256 tokens | 512 tokens (12 % des clauses de test étaient tronquées) |
| Epochs | 2 | 4, en gardant le **meilleur epoch sur la validation** |
| Loss | cross-entropy standard | cross-entropy **pondérée** (poids = 1/√fréquence, classes rares favorisées) |
| Learning rate | 2e-5 | 3e-5, warmup 10 % |
| Précision | fp32 | fp16 (≈ 2× plus rapide sur T4) |

**Règle de décision (fixée avant de regarder le test) :** on adopte v2 seulement si son **F1 macro de validation**
dépasse celui de v1 (**0,7318**, `colab.ipynb` cellule 6). Le jeu de test n'est évalué qu'**une seule fois**, à la fin.

**Avant de lancer :** Exécution → Modifier le type d'exécution → **GPU T4**. Puis Exécution → Tout exécuter.
Durée estimée : 1 h à 1 h 30. Les checkpoints sont sauvés sur Google Drive à chaque epoch :
si Colab se déconnecte, relancer tout reprend l'entraînement là où il s'était arrêté.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), "Pas de GPU : Exécution → Modifier le type d'exécution → GPU T4"

name, memory.total [MiB]
Tesla T4, 15360 MiB


In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/Projet_Deep_Learning")
OUT = DRIVE / "v2"                 # checkpoints, modèle final et prédictions
OUT.mkdir(parents=True, exist_ok=True)
print("Sorties dans :", OUT)

Mounted at /content/drive
Sorties dans : /content/drive/MyDrive/Projet_Deep_Learning/v2


In [ ]:
!pip install -q scikit-learn
import random, numpy as np, torch, transformers
from datasets import load_dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding,
                          Trainer, TrainingArguments)
from sklearn.metrics import accuracy_score, f1_score

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

MODEL_NAME = "nlpaueb/legal-bert-base-uncased"
MAX_LENGTH = 512
EPOCHS = 4
LR = 3e-5
BATCH = 16
V1_VAL = {"accuracy": 0.8424, "f1_macro": 0.7318}     # v1, colab.ipynb cellule 6
print("transformers", transformers.__version__)

transformers 5.20.1


In [ ]:
ledgar = load_dataset("coastalcph/lex_glue", "ledgar")
names = ledgar["train"].features["label"].names
K = len(names)

# Poids de classes : 1/sqrt(fréquence), normalisés à une moyenne de 1.
# Plus doux que 1/fréquence, qui sacrifie trop l'accuracy des classes fréquentes.
counts = np.bincount(ledgar["train"]["label"], minlength=K)
w = 1.0 / np.sqrt(counts)
class_weights = torch.tensor(w / w.mean(), dtype=torch.float)
order = np.argsort(counts)
print(f"{K} classes | poids min {class_weights.min():.2f} ({names[order[-1]]}, {counts[order[-1]]} ex.)"
      f" | poids max {class_weights.max():.2f} ({names[order[0]]}, {counts[order[0]]} ex.)")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


README.md:   0%|          | 0.00/34.1k [00:00<?, ?B/s]

ledgar/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 20.9MB            

ledgar/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

ledgar/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 3.31MB            

...

ledgar/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/60000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10000 [00:00<?, ? examples/s]

100 classes | poids min 0.34 (Governing Laws, 3167 ex.) | poids max 3.99 (Books, 23 ex.)


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH)   # pas de padding ici

tok = ledgar.map(tokenize, batched=True, remove_columns=["text"])
lengths = np.array([len(x) for x in tok["test"]["input_ids"]])
print(f"Test : {np.mean(lengths >= MAX_LENGTH):.1%} des clauses encore tronquées à {MAX_LENGTH} tokens")

config.json:   0%|          | 0.00/1.02k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/222k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Map:   0%|          | 0/60000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Test : 1.4% des clauses encore tronquées à 512 tokens


In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=K, id2label=dict(enumerate(names)), label2id={n: i for i, n in enumerate(names)})


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(-1)
    return {"accuracy": accuracy_score(labels, preds),
            "f1_macro": f1_score(labels, preds, average="macro", zero_division=0)}


class WeightedTrainer(Trainer):
    # Cross-entropy pondérée par classe à la place de la loss par défaut du modèle.
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss = torch.nn.functional.cross_entropy(
            outputs.logits.float(), labels, weight=class_weights.to(outputs.logits.device))
        return (loss, outputs) if return_outputs else loss


steps_per_epoch = int(np.ceil(len(tok["train"]) / BATCH))
extra = {"group_by_length": True} if "group_by_length" in TrainingArguments.__dataclass_fields__ else {}
args = TrainingArguments(
    output_dir=str(OUT / "checkpoints"),
    learning_rate=LR, num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH, per_device_eval_batch_size=64,
    warmup_steps=int(0.1 * steps_per_epoch * EPOCHS), lr_scheduler_type="linear", weight_decay=0.01,
    fp16=True,
    eval_strategy="epoch", save_strategy="epoch", save_total_limit=2,
    load_best_model_at_end=True, metric_for_best_model="f1_macro", greater_is_better=True,
    logging_steps=200, report_to="none", seed=SEED, **extra)

trainer = WeightedTrainer(model=model, args=args,
                          train_dataset=tok["train"], eval_dataset=tok["validation"],
                          processing_class=tokenizer, data_collator=DataCollatorWithPadding(tokenizer),
                          compute_metrics=compute_metrics)
print("group_by_length :", bool(extra), "| steps/epoch :", steps_per_epoch)

[transformers] You passed `num_labels=100` which is incompatible to the `id2label` map of length `2`.


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  440MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: nlpaueb/legal-bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those 

group_by_length : False | steps/epoch : 3750


In [8]:
import time
resume = any((OUT / "checkpoints").glob("checkpoint-*"))
print("Reprise depuis un checkpoint Drive" if resume else "Nouvel entraînement")
t0 = time.time()
trainer.train(resume_from_checkpoint=True if resume else None)
print(f"Entraînement terminé en {(time.time() - t0) / 60:.0f} min")

Nouvel entraînement


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.788526,0.733109,0.835400,0.748603
2,0.550695,0.625396,0.858300,0.784782
3,0.356194,0.591723,0.869400,0.797639


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.788526,0.733109,0.835400,0.748603
2,0.550695,0.625396,0.858300,0.784782
3,0.356194,0.591723,0.869400,0.797639
4,0.307499,0.599091,0.876300,0.805661


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Entraînement terminé en 88 min


In [9]:
# Historique validation epoch par epoch
for h in trainer.state.log_history:
    if "eval_f1_macro" in h:
        print(f"epoch {h['epoch']:.0f} : val accuracy {h['eval_accuracy']:.4f} | val F1 macro {h['eval_f1_macro']:.4f}")

val = trainer.evaluate(tok["validation"])           # meilleur epoch (load_best_model_at_end)
print(f"\nv2 (meilleur epoch) : val accuracy {val['eval_accuracy']:.4f} | val F1 macro {val['eval_f1_macro']:.4f}")
print(f"v1                  : val accuracy {V1_VAL['accuracy']:.4f} | val F1 macro {V1_VAL['f1_macro']:.4f}")
ADOPT_V2 = val["eval_f1_macro"] > V1_VAL["f1_macro"]
print("\nDécision :", "ADOPTER v2" if ADOPT_V2 else "GARDER v1 (v2 ne fait pas mieux sur la validation)")

epoch 1 : val accuracy 0.8354 | val F1 macro 0.7486
epoch 2 : val accuracy 0.8583 | val F1 macro 0.7848
epoch 3 : val accuracy 0.8694 | val F1 macro 0.7976
epoch 4 : val accuracy 0.8763 | val F1 macro 0.8057


Training Loss,Validation Loss,Epoch,Accuracy,F1 Macro
0.307499,0.599091,4,0.876300,0.805661



v2 (meilleur epoch) : val accuracy 0.8763 | val F1 macro 0.8057
v1                  : val accuracy 0.8424 | val F1 macro 0.7318

Décision : ADOPTER v2


In [ ]:
# Évaluation finale sur le test : une seule fois, modèle figé.
pred = trainer.predict(tok["test"])
probs_v2 = torch.softmax(torch.tensor(pred.predictions, dtype=torch.float), -1).numpy()
y_test = np.array(pred.label_ids)
y_pred_v2 = probs_v2.argmax(-1)
f1s = f1_score(y_test, y_pred_v2, average=None, zero_division=0)
print(f"v2 test : accuracy {accuracy_score(y_test, y_pred_v2):.4f} | F1 macro {f1s.mean():.4f}"
      f" | classes à F1 = 0 : {(f1s == 0).sum()}")

np.save(OUT / "test_probs_v2.npy", probs_v2.astype(np.float32))
np.save(OUT / "test_preds_v2.npy", y_pred_v2)
np.save(OUT / "test_labels.npy", y_test)

In [11]:
final = OUT / "bert-ledgar-v2"
trainer.save_model(str(final))
tokenizer.save_pretrained(str(final))
!cd "{OUT}" && zip -qr bert-ledgar-v2.zip bert-ledgar-v2
print("Modèle sauvé :", final, "+ bert-ledgar-v2.zip")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Modèle sauvé : /content/drive/MyDrive/Projet_Deep_Learning/v2/bert-ledgar-v2 + bert-ledgar-v2.zip


## Comparaison avec v1 et DistilBERT (test de McNemar, TP06)

Pour comparer les modèles sur **les mêmes 10 000 clauses**, on recalcule sur GPU (≈ 2 min chacun) les probabilités de :
- **v1** : déposer `bert-ledgar.zip` dans le panneau Fichiers de Colab (`/content/bert-ledgar.zip`),
  ou le dossier dézippé dans `MyDrive/Projet_Deep_Learning/bert-ledgar` ;
- **DistilBERT** `SalmanAbbasi/lexglue-ledgar-distilbert` (référence externe, téléchargée depuis HuggingFace).

v1 et DistilBERT sont évalués à 256 tokens, comme dans le notebook principal, pour reproduire les chiffres déjà rapportés.

In [ ]:
import zipfile

@torch.no_grad()
def predict_probs(source, texts, max_length, batch_size=64):
    tok_ = AutoTokenizer.from_pretrained(source)
    mdl = AutoModelForSequenceClassification.from_pretrained(source).to("cuda").eval()
    out = []
    for i in range(0, len(texts), batch_size):
        enc = tok_(texts[i:i + batch_size], truncation=True, max_length=max_length,
                   padding=True, return_tensors="pt").to("cuda")
        with torch.autocast("cuda", dtype=torch.float16):
            logits = mdl(**enc).logits
        out.append(torch.softmax(logits.float(), -1).cpu().numpy())
    return np.concatenate(out)


test_texts = list(ledgar["test"]["text"])
probs = {"v2 Legal-BERT": probs_v2}

v1_dir = DRIVE / "bert-ledgar"
if not v1_dir.is_dir() and Path("/content/bert-ledgar.zip").exists():
    zipfile.ZipFile("/content/bert-ledgar.zip").extractall("/content")
    v1_dir = Path("/content/bert-ledgar")
if v1_dir.is_dir():
    probs["v1 BERT"] = predict_probs(str(v1_dir), test_texts, 256)
    np.save(OUT / "test_probs_v1.npy", probs["v1 BERT"].astype(np.float32))
else:
    print("v1 introuvable : déposer bert-ledgar.zip dans /content puis relancer cette cellule")

probs["DistilBERT (externe)"] = predict_probs("SalmanAbbasi/lexglue-ledgar-distilbert", test_texts, 256)
np.save(OUT / "test_probs_distilbert.npy", probs["DistilBERT (externe)"].astype(np.float32))

for name, p in probs.items():
    yp = p.argmax(-1)
    f = f1_score(y_test, yp, average=None, zero_division=0)
    print(f"{name:22s} accuracy {accuracy_score(y_test, yp):.4f} | F1 macro {f.mean():.4f} | F1 = 0 : {(f == 0).sum()}")

In [ ]:
import math
from itertools import combinations


def mcnemar(b, c):
    # Test de McNemar avec correction de continuité (TP06, exercice 20).
    if b + c == 0:
        return 0.0, 1.0
    chi2 = (abs(b - c) - 1) ** 2 / (b + c)
    return chi2, math.erfc(math.sqrt(chi2 / 2))


for (na, pa), (nb, pb) in combinations(probs.items(), 2):
    ok_a, ok_b = pa.argmax(-1) == y_test, pb.argmax(-1) == y_test
    b, c = int(np.sum(ok_a & ~ok_b)), int(np.sum(~ok_a & ok_b))
    chi2, p = mcnemar(b, c)
    verdict = "significatif" if p < 0.05 else "non significatif"
    print(f"{na} vs {nb} : seul A juste {b}, seul B juste {c} | chi2 = {chi2:.2f}, p = {p:.2g} → {verdict} à 5 %")

## À récupérer pour le notebook principal

Dans `MyDrive/Projet_Deep_Learning/v2/` :
- `test_probs_v2.npy`, `test_preds_v2.npy`, `test_labels.npy` : prédictions et probabilités (calibration sans ré-inférence CPU) ;
- `test_probs_v1.npy`, `test_probs_distilbert.npy` : pour la comparaison et McNemar ;
- `bert-ledgar-v2.zip` : le modèle (≈ 400 Mo), à dézipper dans `models/bert-ledgar-v2/` en local.